In [52]:
!pip install bitarray
!pip install mmh3


In [53]:
from bitarray import bitarray
import hashlib
from hashlib import sha3_256, sha256, blake2b
import math 
import mmh3
import string
import json
import requests

In [54]:
#asked ChatGPT how large my bitarray needs to be to hold the large dataset
#isn't this very large and will cause hash collisions?

# size = 360000
# bits = bitarray(size)
# bits.setall(0)
#bits n should be larger than data set size

In [55]:
#Asked ChatGPT to explain bitarray library, fixed number of bits, salting, and hashes
#later functions weren't working and asked ChatGPT if I was adding words to the bloom filter right. Modified this code

words = []

with open('words.txt', 'r') as file:
    for line in file:
        word = line.strip()
        words.append(word)
        

In [56]:
class BloomFilter(object):
    #uses murmur3 hash function
    def __init__(self, items_count, fp_prob):
        #items_count number of items expected to be sotred in bloom filter
        #fp_prob false positive probability in decimal
        self.fp_prob = fp_prob #false positive in decimal; optional if size is fixed
        self.size = 3600000 #set size of bitarray
        self.hash_count = self.get_hash_count(self.size, items_count) #number of hash function
        self.bit_array = bitarray(self.size) #bit array of given size
        self.bit_array.setall(0) #initialize all bits as 0
    def add(self, item):
        #add an item in the filter
        digests = []
        for i in range(self.hash_count): #create digest for given item, i seed for mmh3.hash; with different seed digest created is different
            digest = mmh3.hash(item, i) % self.size
            digests.append(digest)
            self.bit_array[digest] = True #set the bit True in bit_array
    def check(self, item):
        #check for item in filteer
        for i in range(self.hash_count):
            digest = mmh3.hash(item, i) % self.size
            if self.bit_array[digest] == False:
                #if any of bit is False, not present; else possibility it exists
                return False
        return True
    def get_hash_count(self, m, n):
        #return hash function for formula
        #m integer size of array
        #n integer number of items expected to be stored
        k = (m/n) * math.log(2)
        return int(k)


# https://www.geeksforgeeks.org/python/bloom-filters-introduction-and-python-implementation/


In [57]:
#for each word in the list, apply all three hash functions and set the corresponding bits in the bitarray.
#All return in [0, size] where size is some integer specified elsewhere

def my_hash(s):
    return int(sha256(s.lower().encode()).hexdigest(), 16) % size
def my_hash2(s):
    return int(blake2b(s.lower().encode()).hexdigest(), 16) % size
def my_hash3(s):
    return int(sha3_256(s.lower().encode()).hexdigest(), 16) % size

for word in words: 
    index1 = my_hash(word)
    index2 = my_hash2(word)
    index3 = my_hash3(word)
    bits[index1] = 1
    bits[index2] = 1
    bits[index3] = 1 #hashes all words in words and set 1 to dictate there is something there
    
#Asked ChatGPT is I answered all parts of the question and modified code (define size) 


In [58]:
#b. Create a function that checks all possible single-character substitutions for a given word using the Bloom filter. 
# Return words flagged by the filter as potential matches. 

#I need the function to take in each word, replace a single character, compare to the rest of list and return if it is a match. Repeat for all letter combinations

#Asked ChatGPT how to make a function that replces single character in given word
def single_char_changes(word): 
    replaced_words = []
    for i in range(len(word)):
        for letter in string.ascii_lowercase:
            if word[i] != letter:
                changed = word[:i] + letter + word[i+1:]
                replaced_words.append(changed)
    return replaced_words

#Tested that the function does single character substitution for word given
print(single_char_changes('cat'))
len(single_char_changes('cat'))


['aat', 'bat', 'dat', 'eat', 'fat', 'gat', 'hat', 'iat', 'jat', 'kat', 'lat', 'mat', 'nat', 'oat', 'pat', 'qat', 'rat', 'sat', 'tat', 'uat', 'vat', 'wat', 'xat', 'yat', 'zat', 'cbt', 'cct', 'cdt', 'cet', 'cft', 'cgt', 'cht', 'cit', 'cjt', 'ckt', 'clt', 'cmt', 'cnt', 'cot', 'cpt', 'cqt', 'crt', 'cst', 'ctt', 'cut', 'cvt', 'cwt', 'cxt', 'cyt', 'czt', 'caa', 'cab', 'cac', 'cad', 'cae', 'caf', 'cag', 'cah', 'cai', 'caj', 'cak', 'cal', 'cam', 'can', 'cao', 'cap', 'caq', 'car', 'cas', 'cau', 'cav', 'caw', 'cax', 'cay', 'caz']


75

In [59]:
#BloomFilter(number of items, false positive)
#https://www.geeksforgeeks.org/python/bloom-filters-introduction-and-python-implementation/
bloomf = BloomFilter(3600000, 0.05)

#add words to BloomFilter
for word in words:
    bloomf.add(word)

#create the function to check if word against bloom filter
def test_in_filter(test):
    return test in words if bloomf.check(test) else False

#test with words known in filter and known not in filter
print(test_in_filter('abandoner'))
print(test_in_filter('carolinejk'))

True
False


In [60]:
#create full function
#Asked ChatGPT how to compare again the bloom filter and ensure function is doing what i want. 
def spell_check(word):
    def single_char_changes(word): 
        replaced_words = []
        for i in range(len(word)):
            for letter in string.ascii_lowercase:
                if word[i] != letter:
                    changed = word[:i] + letter + word[i+1:]
                    replaced_words.append(changed)
        return replaced_words
    candidates = single_char_changes(word)
    for candidate in candidates:
        if candidate in words:
            print(f"{candidate} is a Match!")

In [61]:
#test function
spell_check('cat')

eat is a Match!
qat is a Match!
rat is a Match!
xat is a Match!
yat is a Match!
zat is a Match!
cwt is a Match!
cag is a Match!
caw is a Match!
cay is a Match!


In [62]:
# Implement a function to test how well the Bloom filter suggests corrections. 
# A suggestion list is considered "good" if it contains no more than three suggestions and includes the correct word.

#https://www.geeksforgeeks.org/python/read-json-file-using-python/

#call in typos and visualize
with open('typos.json', 'r') as file2:
    typos = json.load(file2)

In [63]:
# to test how well the bloom filter suggests corrections, i want to build on the function I made
# that function will be nested in another one so that it returns the suggestions and dictates if it's good or not by giving amount of suggestions 
# it needs to know if its correct by having the second word in the pair from typos? No second word needs to be in list. Typos is given to use as testing data
# 
spell_check('wrinkvy')

wrinkly is a Match!


In [64]:
# make list of all words with replace all characters
# for each word in that list, check if its in the bloom
# return amount of words from list that are in bloom (possible suggestions)
# return if correct word is in that list and how many words returned as possible suggestions
# if possible suggestions is more than 3, bad outcome. If less than 3, good outcome

def good_bloom(typed_word, correct_word, bloom):
  def single_char_changes(typed_word): 
        replaced_words_bloom = []
        for i in range(len(typed_word)):
            for letter in string.ascii_lowercase:
                if typed_word[i] != letter:
                    changed = typed_word[:i] + letter + typed_word[i+1:]
                    replaced_words_bloom.append(changed)
        return replaced_words_bloom
  candidates = single_char_changes(typed_word) # puts all single character changes in list as candidates to be a match
  matches = []
  for candidate in candidates: #cycles through each single character change and stores in matches if it is in the word list
    if candidate in words:
        matches.append(candidate)
  print(matches)
  if len(matches) <= 3 and correct_word in matches: #and correct_word in matches: # is matches is more than 3, deems it a bad outcome. If matches is less than 3 deems it a good outcome; need to add if it includes correct word is good, if not is bad. 
      return (f"This is a good outcome, it has {len(matches)} matches and includes the correct word)")
  else:
      return ("This is a bad outcome")
  
  

In [65]:
good_bloom('wrinkvy', 'wrinkly', bloomf) #test with wrinkly which only has 1 match, good outcome

['wrinkly']


'This is a good outcome, it has 1 matches and includes the correct word)'

In [66]:
good_bloom('eat', 'cat', bloomf) #test with eat/cat bad outcome

['qat', 'rat', 'xat', 'yat', 'zat', 'eft', 'ent', 'est', 'ext', 'ead', 'ean', 'ear']


'This is a bad outcome'

In [67]:
# Experiment with different Bloom filter sizes and combinations of 1, 2, or 3 hash functions. 
#create 1 bloom for 1 hash size; 1 bloom for 2 hash size; 1 bloom for 3 hash size

# Asked ChatGPT what to edit to change bit size and hashes

class BloomFilter_new:
    def __init__(self, size, hash_functions): #now uses any hash function
        #items_count number of items expected to be sotred in bloom filter
        self.size = size #set size asspecified in bloom
        self.bit_array = bitarray(self.size) #bit array of given size
        self.bit_array.setall(0) #initialize all bits as 0
        self.hash_functions = hash_functions #allows provided hashing
    def add(self, item):
        for func in self.hash_functions:
            digest = func(item) % self.size #each hash produces integer from item within bounds of bit array
            self.bit_array[digest] = 1 #then sets that bit to 1
    def check(self, item):
        return all(self.bit_array[func(item) % self.size] for func in self.hash_functions) #checks if item is in filter
    
# hashes do not have to change


In [75]:
# hashes do not have to change
# create new blooms
# are these different ways to hash? (sha256, blake2b, sha3_256)

def my_hash(s):
    return int(sha256(s.lower().encode()).hexdigest(), 16) % size
def my_hash2(s):
    return int(blake2b(s.lower().encode()).hexdigest(), 16) % size
def my_hash3(s):
    return int(sha3_256(s.lower().encode()).hexdigest(), 16) % size

for word in words: 
    index1 = my_hash(word)
    index2 = my_hash2(word)
    index3 = my_hash3(word)
    bits[index1] = 1
    bits[index2] = 1
    bits[index3] = 1 #hashes all words in words and set 1 to dictate there is something there
    
#Asked ChatGPT is I answered all parts of the question and modified code (define size) 

bloom1 = BloomFilter_new(10000000,my_hash)
bloom2 = BloomFilter_new(10000000, (my_hash, my_hash2))
bloom3 = BloomFilter_new(10000000,(my_hash, my_hash2, my_hash3))


In [94]:
#test all 3 with flower
good_bloom('floeer', 'flower', bloom1)

['floter']


'This is a bad outcome'

In [95]:
good_bloom('floeer', 'flower', bloom2)


['floter']


'This is a bad outcome'

In [97]:
good_bloom('floeer', 'flower', bloom3)

['floter']


'This is a bad outcome'

In [ ]:
#track output/performance of each bloom 
#asked ChatGPT how to modify to keep track of good outcome and bad outcome

def track_bloom_one(typo_pairs, bloom):
    correct_one_hash = []
    wrong_one_hash = []
    for typed_word, correct_word in typo_pairs:
        result = good_bloom(typed_word, correct_word, bloom)
        if "good" in result.lower():
            correct_one_hash.append('good')
        else:
            wrong_one_hash.append('bad')
    return correct_one_hash, wrong_one_hash

In [91]:
typo_pairs = [('floeer', 'flower')]

track_bloom_one(typo_pairs, bloom1)

['floter']


([], ['bad'])

In [ ]:
# Track the rate of false positives (incorrect words misidentified as correct) and good suggestions (as defined above). 
# Plot the results.
# Approximate how many bits are necessary to acheive 85% good suggestions with each combination of 1, 2, 3, hashes

In [ ]:
#TA Questions:
# implementation check of bloom filter not working
# how to download human genome 
# how big should my bitarray size be
# how can I be sure my alf2 merge workds/testing
# advice for generating fake data